In [ ]:
# ============================================
# MODEL EXPLAINABILITY WITH SHAP
# ============================================

import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
plt.style.use('seaborn-v0_8-darkgrid')

print("="*70)
print("TASK 3: MODEL EXPLAINABILITY")
print("="*70)

### 1. LOAD MODELS AND DATA

In [ ]:
print("\n1. Loading models and data...")

# Load best model from Task 2
best_model = joblib.load('models/xgboost_ecomm.pkl')
print(f"Model loaded: {type(best_model).__name__}")

# Load processed data
fraud_data = pd.read_csv('data/processed/fraud_data_engineered.csv')

# Load feature names
feature_names = joblib.load('models/feature_names_ecomm.pkl')

# Prepare features (same as in Task 2)
excluded_features = ['user_id', 'signup_time', 'purchase_time', 'ip_address', 'class']
X_ecomm = fraud_data.drop(excluded_features, axis=1, errors='ignore')

# One-hot encode categorical variables
categorical_cols = X_ecomm.select_dtypes(include=['object', 'category']).columns
X_ecomm_encoded = pd.get_dummies(X_ecomm, columns=categorical_cols, drop_first=True)

# Align with training features
missing_cols = set(feature_names) - set(X_ecomm_encoded.columns)
for col in missing_cols:
    X_ecomm_encoded[col] = 0
X_ecomm_encoded = X_ecomm_encoded[feature_names]

y_ecomm = fraud_data['class']

print(f"✅ Features: {X_ecomm_encoded.shape[1]}")
print(f"✅ Samples: {X_ecomm_encoded.shape[0]}")

### 2. INITIALIZE SHAP EXPLAINER

In [ ]:
print("\n" + "="*70)
print("2. SHAP ANALYSIS INITIALIZATION")
print("="*70)

from src.explainability import SHAPExplainer

# Use sample for faster computation
sample_size = 2000
if len(X_ecomm_encoded) > sample_size:
    sample_indices = np.random.choice(len(X_ecomm_encoded), sample_size, replace=False)
    X_sample = X_ecomm_encoded.iloc[sample_indices].values
    y_sample = y_ecomm.iloc[sample_indices].values
else:
    X_sample = X_ecomm_encoded.values
    y_sample = y_ecomm.values

print(f"Using {len(X_sample)} samples for SHAP analysis")

# Initialize SHAP explainer
shap_explainer = SHAPExplainer(
    model=best_model,
    X=X_sample,
    feature_names=feature_names,
    model_type='tree'  # Change to 'linear' for logistic regression
)


### 3. GLOBAL FEATURE IMPORTANCE

In [ ]:
print("\n" + "="*70)
print("3. GLOBAL FEATURE IMPORTANCE")
print("="*70)

# Get SHAP-based feature importance
shap_importance_df = shap_explainer.global_feature_importance()

print("\nTop 10 Most Important Features (SHAP):")
print("-"*50)
for i, (_, row) in enumerate(shap_importance_df.head(10).iterrows(), 1):
    print(f"{i:2}. {row['feature']:30} {row['shap_importance']:.4f}")

# Compare with model's built-in feature importance
if hasattr(best_model, 'feature_importances_'):
    print("\nComparing with model's built-in feature importance:")
    print("-"*50)
    
    model_importance = pd.DataFrame({
        'feature': feature_names,
        'model_importance': best_model.feature_importances_
    }).sort_values('model_importance', ascending=False)
    
    # Merge with SHAP importance
    comparison_df = pd.merge(
        shap_importance_df.head(20),
        model_importance,
        on='feature',
        how='left'
    )
    
    # Calculate correlation
    correlation = comparison_df['shap_importance'].corr(comparison_df['model_importance'])
    print(f"Correlation between SHAP and model importance: {correlation:.3f}")
    
    # Display top features from both
    print("\nTop 5 features comparison:")
    for i, (_, row) in enumerate(comparison_df.head(5).iterrows(), 1):
        print(f"{i}. {row['feature']:30} SHAP: {row['shap_importance']:.4f} Model: {row['model_importance']:.4f}")

### 4. SHAP SUMMARY PLOT

In [ ]:
print("\n" + "="*70)
print("4. SHAP SUMMARY PLOT")
print("="*70)

# Create SHAP summary plot
shap_explainer.plot_summary(
    max_features=15,
    save_path='reports/figures/shap_plots/shap_summary.png'
)

### 5. INDIVIDUAL PREDICTION ANALYSIS

In [ ]:
print("\n" + "="*70)
print("5. INDIVIDUAL PREDICTION ANALYSIS")
print("="*70)

# Get predictions
y_pred = best_model.predict(X_ecomm_encoded.values)
y_pred_proba = best_model.predict_proba(X_ecomm_encoded.values)[:, 1]

# Analyze different prediction types
prediction_types = shap_explainer.analyze_predictions(y_ecomm.values, y_pred)

# Analyze examples of each type
print("\nAnalyzing example predictions...")

# True Positive example
if len(prediction_types['true_positives']) > 0:
    tp_idx = prediction_types['true_positives'][0]
    print(f"\n📈 TRUE POSITIVE Example (Index {tp_idx}):")
    print(f"   Actual: Fraud, Predicted: Fraud")
    print(f"   Probability: {y_pred_proba[tp_idx]:.3f}")
    
    tp_analysis = shap_explainer.plot_waterfall(
        tp_idx,
        max_features=10,
        save_path='reports/figures/shap_plots/true_positive_waterfall.png'
    )
    
    # Also create force plot
    shap_explainer.plot_force(
        tp_idx,
        save_path='reports/figures/shap_plots/true_positive_force.html'
    )

# False Positive example
if len(prediction_types['false_positives']) > 0:
    fp_idx = prediction_types['false_positives'][0]
    print(f"\n⚠️ FALSE POSITIVE Example (Index {fp_idx}):")
    print(f"   Actual: Legitimate, Predicted: Fraud")
    print(f"   Probability: {y_pred_proba[fp_idx]:.3f}")
    
    fp_analysis = shap_explainer.plot_waterfall(
        fp_idx,
        max_features=10,
        save_path='reports/figures/shap_plots/false_positive_waterfall.png'
    )

# False Negative example
if len(prediction_types['false_negatives']) > 0:
    fn_idx = prediction_types['false_negatives'][0]
    print(f"\n❌ FALSE NEGATIVE Example (Index {fn_idx}):")
    print(f"   Actual: Fraud, Predicted: Legitimate")
    print(f"   Probability: {y_pred_proba[fn_idx]:.3f}")
    
    fn_analysis = shap_explainer.plot_waterfall(
        fn_idx,
        max_features=10,
        save_path='reports/figures/shap_plots/false_negative_waterfall.png'
    )


### 6. DEPENDENCE PLOTS

In [ ]:
print("\n" + "="*70)
print("6. FEATURE DEPENDENCE ANALYSIS")
print("="*70)

# Analyze dependence for top features
top_features = shap_importance_df.head(3)['feature'].tolist()

for feature in top_features:
    print(f"\nAnalyzing dependence for: {feature}")
    
    # Find potential interaction feature
    potential_interactions = [f for f in shap_importance_df['feature'].tolist() 
                             if f != feature and any(keyword in f for keyword in ['time', 'value', 'country'])]
    
    interaction_feature = potential_interactions[0] if potential_interactions else None
    
    shap_explainer.plot_dependence(
        feature,
        interaction_feature=interaction_feature,
        save_path=f'reports/figures/shap_plots/dependence_{feature}.png'
    )




### 7. BUSINESS RECOMMENDATIONS

In [ ]:
print("\n" + "="*70)
print("7. BUSINESS RECOMMENDATIONS")
print("="*70)

from src.business_recommendations import BusinessRecommendations

# Load model performance metrics
model_results = joblib.load('models/model_results_summary.pkl')
best_model_performance = model_results['xgboost']  # Or your best model

# Define feature descriptions for better insights
feature_descriptions = {
    'time_since_signup': 'Hours since user account creation',
    'purchase_value': 'Transaction amount in dollars',
    'transactions_last_1h': 'Number of transactions in last 1 hour',
    'transactions_last_24h': 'Number of transactions in last 24 hours',
    'hour_of_day': 'Hour of day when transaction occurred',
    'day_of_week': 'Day of week (0=Monday, 6=Sunday)',
    'country_US': 'Transaction originated from United States',
    'country_GB': 'Transaction originated from United Kingdom',
    'source_Ads': 'User came from advertisement',
    'browser_Chrome': 'User used Chrome browser',
    'sex_M': 'User is male',
    'age': 'User age',
    'purchase_value_category_high': 'High value transaction category',
    'purchase_value_category_medium': 'Medium value transaction category',
    'is_weekend': 'Transaction occurred on weekend',
    'users_per_device': 'Number of users sharing same device',
    'transactions_per_device': 'Total transactions from same device'
}

# Generate business recommendations
business_analyst = BusinessRecommendations(shap_importance_df, feature_descriptions)

# Get top drivers
top_drivers = business_analyst.get_top_drivers(10)
print("\nTop 10 Fraud Prediction Drivers:")
print(top_drivers[['feature', 'description', 'importance']].round(4))

# Generate recommendations
recommendations = business_analyst.generate_recommendations(
    shap_analysis_results=shap_explainer,
    model_performance=best_model_performance,
    n_recommendations=5
)

print("\nActionable Business Recommendations:")
print("="*60)
for i, rec in recommendations.iterrows():
    print(f"\n{i+1}. {rec['title']}")
    print(f"   📝 {rec['description']}")
    print(f"   🎯 Action: {rec['action']}")
    print(f"   📈 Expected Impact: {rec['expected_impact']}")
    print(f"   ⚙️  Effort: {rec['implementation_effort']}")
    print(f"   🚨 Priority: {rec['priority']}")

# Generate implementation roadmap
roadmap = business_analyst.generate_implementation_roadmap(recommendations)
print("\nImplementation Roadmap:")
print(roadmap[['recommendation', 'priority', 'timeline', 'owner']])

# Generate executive summary
executive_summary = business_analyst.create_executive_summary(
    top_drivers,
    recommendations,
    best_model_performance
)

print("\n" + "="*70)
print("EXECUTIVE SUMMARY")
print("="*70)
print(executive_summary)

# Save executive summary
with open('reports/executive_summary.txt', 'w') as f:
    f.write(executive_summary)

### 8. SAVE ANALYSIS RESULTS

In [ ]:
print("\n" + "="*70)
print("10. SAVING ANALYSIS RESULTS")
print("="*70)

# Save SHAP importance
shap_importance_df.to_csv('reports/shap_feature_importance.csv', index=False)

# Save recommendations
recommendations.to_csv('reports/business_recommendations.csv', index=False)
roadmap.to_csv('reports/implementation_roadmap.csv', index=False)

# Save comprehensive report
report_data = {
    'shap_importance': shap_importance_df.head(20).to_dict(),
    'recommendations': recommendations.to_dict(),
    'model_performance': best_model_performance,
    'top_drivers': top_drivers.to_dict(),
    'surprising_findings': surprising_findings
}

joblib.dump(report_data, 'reports/shap_analysis_report.pkl')

print("\n✅ Analysis results saved to 'reports/' directory")
print("✅ Visualizations saved to 'reports/figures/shap_plots/'")
print("✅ Business recommendations generated")
print("✅ Executive summary created")

print("\n" + "="*70)
print("TASK 3 COMPLETED SUCCESSFULLY!")
print("="*70)